# LangGraph Intermediate — Advanced Agents on SAP BTP

Builds on **Notebook 1 (LangGraph Fundamentals)**. Every section is mockable — no SAP credentials required.

## Roadmap

| # | Section | Key Concept | SAP Relevance |
|---|---------|-------------|---------------|
| 1 | Multi-agent: Supervisor | Orchestrate specialist sub-agents | Procurement · maintenance · HR workflows |
| 2 | Subgraphs | Composable, reusable agent modules | SAP domain-specific building blocks |
| 3 | Parallel Execution (Send API) | Fan-out / map-reduce | Query multiple SAP systems simultaneously |
| 4 | Advanced State & Custom Reducers | Private state · merge strategies | Multi-agent state isolation |
| 5 | RAG Tool Node with HANA (mocked) | `langchain-hana` · `HanaDB` pattern | HANA Cloud Vector Engine for grounding |
| 6 | Long-term Memory | Cross-thread semantic memory | Persist user context across sessions |
| 7 | Reflection / Self-critique Loop | Quality gate pattern | Validate before writing back to SAP |
| 8 | Observability with Langfuse | BTP-friendly production tracing | Audit trail · debugging · cost tracking |

---

## Section 1 — Multi-agent: Supervisor Pattern

For complex SAP workflows (procurement, HR, maintenance) you often need **multiple specialist agents** coordinated by a **supervisor**:

```
START → supervisor ─┬─► sales_agent   ──┐
                    ├─► ticket_agent  ──┤ (all loop back to supervisor)
                    └─► general_agent ──┘
                    └─► FINISH → END
```

- The **supervisor** is an LLM node: it reads the conversation and picks the next worker, or says FINISH.
- Each **worker** is a self-contained `create_agent` with its own tools.
- After every worker runs, control returns to the supervisor — it can call multiple agents in sequence.

This pattern is used in the SAP community for maintenance notification creation, procurement workflows, and HR request handling (multi-agent use case blogs, QuickLaunch series Part 5+).

### 1.1 Setup

In [ ]:
from importlib.metadata import version, PackageNotFoundError
import os
from dotenv import load_dotenv

# Load variables from .env in the project root
load_dotenv()

packages = {
    'langgraph':        'langgraph',
    'langchain_core':   'langchain-core',
    'langchain_openai': 'langchain-openai',
    'langfuse':         'langfuse',           # Section 8
    'langchain_hana':   'langchain-hana',     # Section 5 (optional)
    'sap-ai-sdk-gen':   'sap-ai-sdk-gen',     # SAP BTP (Notebook 1, Section 9)
}

for display, pkg in packages.items():
    try:
        print(f'{display:30s} {version(pkg)}')
    except PackageNotFoundError:
        print(f'{display:30s} NOT INSTALLED')

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found. "
        "Add it to your .env file: OPENAI_API_KEY=sk-..."
    )

# ── LLM setup ─────────────────────────────────────────────────────────────
# For SAP BTP: swap for gen_ai_hub (see Notebook 1, Section 9)
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('\nLLM ready.')

### 1.2 Specialist Agents

Each specialist agent knows only the tools relevant to its domain.
Using `create_agent` (from `langchain.agents`, same as Notebook 1, Section 6) keeps this concise.

In [ ]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.agents import create_agent


# ── SAP domain tools ───────────────────────────────────────────────────────
@tool
def lookup_sales_order(order_id: str) -> str:
    'Look up a SAP sales order by its ID and return order details.'
    orders = {
        'SO-1001': 'Customer: ACME Corp, Amount: 12,500 EUR, Status: Open',
        'SO-1002': 'Customer: SAP SE, Amount: 87,000 EUR, Status: Delivered',
    }
    return orders.get(order_id, f'Order {order_id} not found.')


@tool
def list_open_tickets(priority: str = 'all') -> str:
    'List open support tickets. Priority: high / medium / low / all.'
    tickets = [
        {'id': 'TKT-001', 'priority': 'high',   'title': 'System outage'},
        {'id': 'TKT-002', 'priority': 'medium', 'title': 'Performance issue'},
        {'id': 'TKT-003', 'priority': 'low',    'title': 'UI display bug'},
    ]
    filtered = [t for t in tickets if priority == 'all' or t['priority'] == priority]
    return '\n'.join(f"{t['id']} [{t['priority']}]: {t['title']}" for t in filtered)


@tool
def calculator(expression: str) -> str:
    'Evaluate a mathematical expression. Example: 3 ** 4 + 12'
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'Error: {e}'


# ── Specialist agents (each owns only its relevant tools) ──────────────────
sales_agent   = create_agent(model=llm, tools=[lookup_sales_order])
ticket_agent  = create_agent(model=llm, tools=[list_open_tickets])
general_agent = create_agent(model=llm, tools=[calculator])

print('Specialist agents ready.')

### 1.3 Supervisor Node

The supervisor is a plain LLM call. It reads the conversation and outputs the name of the next worker — or `FINISH` when done.

The **worker wrapper** translates between the supervisor's state (which has an extra `next` field) and the worker agent's expected state (just `messages`).

In [ ]:
# ── Shared state for the whole multi-agent graph ───────────────────────────
class SupervisorState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next: str   # which agent runs next


WORKERS = ['sales_agent', 'ticket_agent', 'general_agent']


# ── Supervisor node ────────────────────────────────────────────────────────
def supervisor_node(state: SupervisorState) -> dict:
    workers_list = ', '.join(WORKERS)
    system = SystemMessage(content=(
        'You are a supervisor routing SAP requests to the right specialist.\n'
        f'Workers available: {workers_list}\n\n'
        '  sales_agent   — sales orders, order status, customer data\n'
        '  ticket_agent  — support tickets, incidents, service requests\n'
        '  general_agent — calculations and everything else\n\n'
        'Reply with ONLY the worker name (one word) or FINISH when done.'
    ))
    response = llm.invoke([system] + state['messages'])
    choice   = response.content.strip()

    # Normalise: exact match → substring match → FINISH
    next_step = choice if choice in WORKERS else next(
        (w for w in WORKERS if w in choice.lower()), 'FINISH'
    )
    print(f'[supervisor] → {next_step}')
    return {'next': next_step}


# ── Worker wrapper: translate SupervisorState → worker → back ──────────────
def make_worker_node(agent, name: str):
    def _node(state: SupervisorState) -> dict:
        # Pass only messages to avoid state-schema conflicts
        result = agent.invoke({'messages': state['messages']})
        last   = result['messages'][-1]
        print(f'[{name}] {last.content[:80]}')
        return {'messages': [AIMessage(content=last.content, name=name)]}
    return _node


print('Supervisor and worker wrappers defined.')

### 1.4 Assemble and Test

In [ ]:
builder = StateGraph(SupervisorState)

builder.add_node('supervisor',    supervisor_node)
builder.add_node('sales_agent',   make_worker_node(sales_agent,   'sales_agent'))
builder.add_node('ticket_agent',  make_worker_node(ticket_agent,  'ticket_agent'))
builder.add_node('general_agent', make_worker_node(general_agent, 'general_agent'))

builder.add_edge(START, 'supervisor')

# Supervisor decides next step via conditional edge
builder.add_conditional_edges(
    'supervisor',
    lambda state: state['next'],
    {'sales_agent':   'sales_agent',
     'ticket_agent':  'ticket_agent',
     'general_agent': 'general_agent',
     'FINISH':        END}
)

# After each worker → return to supervisor
for worker in WORKERS:
    builder.add_edge(worker, 'supervisor')

multi_agent = builder.compile()

print('Multi-agent supervisor graph compiled.')
try:
    print(multi_agent.get_graph().draw_ascii())
except Exception:
    pass

In [ ]:
def run_multi_agent(question: str):
    print(f'\n{"="*60}')
    print(f'User: {question}')
    result = multi_agent.invoke({'messages': [HumanMessage(content=question)]})
    # The last AIMessage with a name tag came from a worker
    for msg in reversed(result['messages']):
        if isinstance(msg, AIMessage) and getattr(msg, 'name', None):
            print(f'Answer ({msg.name}): {msg.content[:200]}')
            return


run_multi_agent('What is the status of sales order SO-1001?')
run_multi_agent('Show me all high-priority tickets.')
run_multi_agent('What is 17 * 83?')

---
## Section 2 — Subgraphs

A **subgraph** is a compiled `StateGraph` embedded as a single node inside a parent graph.

```
Parent graph:
START → prepare → [enrichment_subgraph] → respond → END

Enrichment subgraph (internal):
START → detect_entity → fetch_enrichment → END
```

Benefits:
- **Encapsulation** — complex multi-step logic becomes a black box
- **Reusability** — plug the same subgraph into multiple parent graphs
- **Testability** — the subgraph can be unit-tested independently

Key rule: the subgraph has its **own state schema**. A wrapper node translates between parent and subgraph states.

### 2.1 Define the Subgraph

In [ ]:
# ── Subgraph state (its own schema, isolated from the parent) ──────────────
class EnrichmentState(TypedDict):
    raw_text:      str   # input from parent
    entity_type:   str   # detected by first node
    enriched_data: str   # output back to parent


def detect_entity(state: EnrichmentState) -> dict:
    text = state['raw_text'].lower()
    if any(kw in text for kw in ['order', 'so-']):
        etype = 'sales_order'
    elif any(kw in text for kw in ['ticket', 'tkt-', 'incident']):
        etype = 'ticket'
    else:
        etype = 'unknown'
    print(f'  [enrich/detect] entity_type = {etype}')
    return {'entity_type': etype}


def fetch_enrichment(state: EnrichmentState) -> dict:
    lookup = {
        'sales_order': 'S/4HANA: order metadata, partner info, incoterms loaded.',
        'ticket':      'Service Cloud: SLA timer, category, assignee loaded.',
        'unknown':     'No SAP entity identified — using generic context.',
    }
    data = lookup[state['entity_type']]
    print(f'  [enrich/fetch] {data[:60]}')
    return {'enriched_data': data}


# ── Compile the subgraph ───────────────────────────────────────────────────
enrich_builder = StateGraph(EnrichmentState)
enrich_builder.add_node('detect', detect_entity)
enrich_builder.add_node('fetch',  fetch_enrichment)
enrich_builder.add_edge(START, 'detect')
enrich_builder.add_edge('detect', 'fetch')
enrich_builder.add_edge('fetch', END)

enrichment_subgraph = enrich_builder.compile()

print('Enrichment subgraph compiled.')
print(enrichment_subgraph.get_graph().draw_ascii())

### 2.2 Embed Subgraph in a Parent Graph

The parent wraps the subgraph in a regular node function. From the parent's perspective it's just another step — the internal logic is hidden.

In [ ]:
class ParentState(TypedDict):
    messages:      Annotated[List[BaseMessage], add_messages]
    raw_text:      str
    enriched_data: str


# ── Wrapper: bridges parent ↔ subgraph state schemas ──────────────────────
def run_enrichment(state: ParentState) -> dict:
    sub_in  = {'raw_text': state['raw_text'], 'entity_type': '', 'enriched_data': ''}
    sub_out = enrichment_subgraph.invoke(sub_in)
    return {'enriched_data': sub_out['enriched_data']}


def prepare(state: ParentState) -> dict:
    text = state['messages'][-1].content
    print(f'[prepare] "{text[:60]}"')
    return {'raw_text': text}


def respond(state: ParentState) -> dict:
    reply = f'Request processed.\n\nContext loaded: {state["enriched_data"]}\n\n(Mocked LLM response.)'
    return {'messages': [AIMessage(content=reply)]}


parent_builder = StateGraph(ParentState)
parent_builder.add_node('prepare', prepare)
parent_builder.add_node('enrich',  run_enrichment)   # ← subgraph is just a node
parent_builder.add_node('respond', respond)
parent_builder.add_edge(START,     'prepare')
parent_builder.add_edge('prepare', 'enrich')
parent_builder.add_edge('enrich',  'respond')
parent_builder.add_edge('respond', END)

parent_graph = parent_builder.compile()

# ── Test all three paths ───────────────────────────────────────────────────
for q in ['Check sales order SO-1001', 'Any open tickets today?', 'What is the BTP pricing?']:
    print(f'\nQ: {q}')
    r = parent_graph.invoke({'messages': [HumanMessage(content=q)], 'raw_text': '', 'enriched_data': ''})
    print(f'A: {r["messages"][-1].content[:120]}')

---
## Section 3 — Parallel Execution: the Send API

The `Send` API lets you **dynamically spawn multiple copies of a node** at runtime — one per item in a list. All copies run in parallel; their outputs are merged by the reducer.

This is LangGraph's **map-reduce** primitive:

```
START ──(Send×N)──► query_sap_system (runs N times in parallel)
                         │
                    results merged by operator.add
                         │
                        END
```

**SAP use case:** query S/4HANA, SuccessFactors, and Service Cloud simultaneously, then combine the results. The SAP community explicitly calls out parallelisation as a key reason to choose LangGraph over plain LangChain.

Key mechanics:
- The fan-out function returns `[Send('node_name', state_for_that_copy), ...]` instead of a string
- Each `Send` can carry different state (which system to query, which query to run, etc.)
- The worker's output key (`results`) merges into the parent state via the `operator.add` reducer

In [ ]:
import operator
from langgraph.types import Send   # ← key import for parallel execution


# ── Outer (aggregating) state ──────────────────────────────────────────────
class ParallelState(TypedDict):
    query:   str
    systems: List[str]
    results: Annotated[List[str], operator.add]   # reducer: concat lists from all workers


# ── Per-worker input state ─────────────────────────────────────────────────
class SystemQueryState(TypedDict):
    query:   str
    system:  str
    results: Annotated[List[str], operator.add]


def query_sap_system(state: SystemQueryState) -> dict:
    'Query a single SAP system (mocked). In production: OData / REST call.'
    mock = {
        'S4HANA':         'S/4HANA: 3 matching orders totalling 45,000 EUR.',
        'SuccessFactors': 'SuccessFactors: 12 employees with matching skills.',
        'ServiceCloud':   'Service Cloud: 2 open high-priority tickets.',
    }
    response = mock.get(state['system'], f'No data from {state["system"]}.')
    print(f'  [worker/{state["system"]}] {response}')
    return {'results': [f'[{state["system"]}] {response}']}


# ── Fan-out function ───────────────────────────────────────────────────────
# Called as a conditional edge. Returns Send objects instead of a node name.
def fan_out(state: ParallelState):
    print(f'[fan_out] spawning {len(state["systems"])} parallel workers')
    return [
        Send('query_sap_system', {'query': state['query'], 'system': s, 'results': []})
        for s in state['systems']
    ]


# ── Build graph ────────────────────────────────────────────────────────────
builder = StateGraph(ParallelState)
builder.add_node('query_sap_system', query_sap_system)

# Conditional edge from START: fan_out returns Sends → LangGraph runs them in parallel
builder.add_conditional_edges(START, fan_out, ['query_sap_system'])
builder.add_edge('query_sap_system', END)

parallel_graph = builder.compile()

# ── Run ────────────────────────────────────────────────────────────────────
result = parallel_graph.invoke({
    'query':   'quarterly revenue impact',
    'systems': ['S4HANA', 'SuccessFactors', 'ServiceCloud'],
    'results': [],
})

print('\n=== Aggregated results (all three systems, merged by operator.add) ===')
for r in result['results']:
    print(r)

---
## Section 4 — Advanced State & Custom Reducers

### 4.1 Reducers control how fields are merged

| Pattern | Code | Behaviour |
|---------|------|-----------|
| Last write wins | `field: str` | Each write replaces the previous value |
| Append list | `field: Annotated[List, operator.add]` | Writes are concatenated |
| Append messages | `field: Annotated[List, add_messages]` | Deduplicates by message id |
| Custom logic | `field: Annotated[T, my_fn]` | Any function `(current, update) → merged` |

### 4.2 Input / Output schemas for clean public APIs

When building reusable agent nodes you often want **scratch fields** (intermediate data, debugging info) that should not be visible to the graph consumer:

```python
graph = StateGraph(FullInternalState, input=InputState, output=OutputState)
```

This is especially useful in the supervisor pattern — workers can have private fields that never leak into the final output.

In [ ]:
# ── Custom reducer: keep the highest-priority item ─────────────────────────
def keep_highest_priority(current: dict | None, update: dict) -> dict:
    'Merge two dicts; always keep the one with the higher SAP priority.'
    if current is None:
        return update
    rank = {'high': 3, 'medium': 2, 'low': 1}
    current_rank = rank.get(current.get('priority', 'low'), 1)
    update_rank  = rank.get(update.get('priority',  'low'), 1)
    return update if update_rank > current_rank else current


class ReducerState(TypedDict):
    items:    Annotated[List[str], operator.add]           # append on every write
    top_item: Annotated[dict,      keep_highest_priority]  # custom: keep highest priority
    counter:  int                                           # last-write-wins (no annotation)


def node_a(state: ReducerState) -> dict:
    return {
        'items':    ['item_from_a'],
        'top_item': {'name': 'Task A', 'priority': 'medium'},
        'counter':  1,
    }


def node_b(state: ReducerState) -> dict:
    return {
        'items':    ['item_from_b'],
        'top_item': {'name': 'Task B', 'priority': 'high'},   # higher — should win
        'counter':  2,
    }


builder = StateGraph(ReducerState)
builder.add_node('a', node_a)
builder.add_node('b', node_b)
builder.add_edge(START, 'a')
builder.add_edge('a', 'b')
builder.add_edge('b', END)

g = builder.compile()
result = g.invoke({'items': [], 'top_item': {}, 'counter': 0})

print('items (operator.add — appended by both):',   result['items'])
print('top_item (custom — highest priority wins):', result['top_item'])
print('counter (no reducer — last-write-wins):',    result['counter'])

In [ ]:
# ── Input / output schemas: hide internal scratch fields ──────────────────

class InputState(TypedDict):
    user_query: str                # what the consumer provides

class OutputState(TypedDict):
    final_answer: str              # what the consumer receives

class InternalState(InputState, OutputState):
    classified_intent: str         # scratch — invisible externally
    raw_data:          str         # scratch — invisible externally


def classify(state: InternalState) -> dict:
    intent = 'order' if 'order' in state['user_query'].lower() else 'general'
    print(f'[classify] intent = {intent}')
    return {'classified_intent': intent}


def fetch(state: InternalState) -> dict:
    if state['classified_intent'] == 'order':
        raw = 'S/4HANA: SO-1001 open, 12,500 EUR.'
    else:
        raw = 'No specific SAP data found.'
    print(f'[fetch] raw_data = {raw!r}')
    return {'raw_data': raw}


def format_answer(state: InternalState) -> dict:
    return {'final_answer': f'Answer: {state["raw_data"]}'}


# input= and output= restrict what's visible at graph boundaries
scoped = StateGraph(InternalState, input=InputState, output=OutputState)
scoped.add_node('classify',      classify)
scoped.add_node('fetch',         fetch)
scoped.add_node('format_answer', format_answer)
scoped.add_edge(START, 'classify')
scoped.add_edge('classify', 'fetch')
scoped.add_edge('fetch', 'format_answer')
scoped.add_edge('format_answer', END)

g2 = scoped.compile()

result = g2.invoke({'user_query': 'What is the status of order SO-1001?'})
print('\nOutput visible to graph consumer:')
print(result)   # only final_answer — classified_intent and raw_data are hidden

---
## Section 5 — RAG Tool Node with SAP HANA (mocked)

RAG (Retrieval-Augmented Generation) grounds agent responses in data stored in a **vector database**.

SAP officially supports this via **`langchain-hana`** (`pip install langchain-hana`), which provides `HanaDB` — a LangChain `VectorStore` backed by the SAP HANA Cloud Vector Engine.

```
Agent
  └─► @tool search_sap_kb(query)
           │
           ▼
      HanaDB.similarity_search(query, k=3)
           │
           ▼
      SAP HANA Cloud — VECTOR_EMBEDDING() native function
           │
           ▼
      Retrieved chunks → injected into LLM context
```

In this section we **mock** the HANA connection with keyword overlap. The structure is identical to production — only the connection line changes.

| Class | Source | Purpose |
|-------|--------|---------|
| `HanaDB` | `langchain-hana` (official SAP) | LangChain VectorStore backed by HANA Cloud |
| `HanaInternalEmbeddings` | `langchain-hana` | Uses HANA's native `VECTOR_EMBEDDING()` — no external embedding model |
| `HanaDB.as_retriever()` | `langchain-hana` | Converts the store to a LangChain retriever |

In [ ]:
from langchain_core.documents import Document


# ── Mock SAP HANA knowledge base ──────────────────────────────────────────
# Production replacement:
#   from langchain_hana import HanaDB, HanaInternalEmbeddings
#   import hdbcli.dbapi as dbapi
#   conn = dbapi.connect(address='<host>', port=443, user='<user>', password='<pw>')
#   vectorstore = HanaDB(connection=conn, embedding=HanaInternalEmbeddings(conn), table_name='SAP_KB')
#   docs = vectorstore.similarity_search(query, k=3)

SAP_KB = [
    Document(
        page_content='Sales order SO-1001: Customer ACME Corp, 12,500 EUR, status Open, created 2024-01-15.',
        metadata={'source': 'S4HANA'}
    ),
    Document(
        page_content='Sales order SO-1002: Customer SAP SE, 87,000 EUR, status Delivered, shipping complete.',
        metadata={'source': 'S4HANA'}
    ),
    Document(
        page_content='Ticket TKT-001: System outage reported, high priority, assigned to ops team, ETA 2h.',
        metadata={'source': 'ServiceCloud'}
    ),
    Document(
        page_content='Ticket TKT-002: BTP performance degradation, medium priority, under investigation.',
        metadata={'source': 'ServiceCloud'}
    ),
    Document(
        page_content='SAP BTP AI Core: provides managed LLM deployments, vector search via HANA, and AI pipelines.',
        metadata={'source': 'BTPCatalog'}
    ),
]


class MockHanaDB:
    'Drop-in mock for HanaDB. Replace with real HanaDB for production.'

    def __init__(self, documents: list):
        self.documents = documents

    def similarity_search(self, query: str, k: int = 2) -> list:
        'Return top-k docs by keyword overlap (stands in for real vector similarity).'
        words = set(query.lower().split())
        scored = [(len(words & set(d.page_content.lower().split())), d) for d in self.documents]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [d for _, d in scored[:k]]


vectorstore = MockHanaDB(SAP_KB)


# ── RAG retrieval tool ─────────────────────────────────────────────────────
@tool
def search_sap_kb(query: str) -> str:
    'Search the SAP knowledge base (backed by HANA Cloud Vector Engine) for relevant context.'
    docs = vectorstore.similarity_search(query, k=2)
    if not docs:
        return 'No relevant documents found.'
    return '\n\n'.join(f'[{d.metadata["source"]}] {d.page_content}' for d in docs)


# ── Smoke test ─────────────────────────────────────────────────────────────
print(search_sap_kb.invoke({'query': 'ACME sales order'}))
print('---')
print(search_sap_kb.invoke({'query': 'high priority system outage ticket'}))

In [ ]:
from langchain.agents import create_agent

# The RAG tool is just another tool in a standard ReAct agent
rag_agent = create_agent(model=llm, tools=[search_sap_kb, calculator])

response = rag_agent.invoke({
    'messages': [HumanMessage(content='Tell me about the ACME order and any related tickets.')]
})

print('=== RAG Agent — conversation trace ===')
for msg in response['messages']:
    role = msg.__class__.__name__.replace('Message', '')
    body = msg.content or str(getattr(msg, 'tool_calls', ''))
    print(f'[{role:12s}] {body[:180]}')

---
## Section 6 — Long-term Memory

Notebook 1 covered **short-term memory**: `MemorySaver` keeps conversation history within one `thread_id`. It resets when the process restarts — and it doesn't span multiple threads.

**Long-term memory** persists *across* thread IDs and process restarts. LangGraph adds a `Store` alongside the checkpointer:

```
Checkpointer (MemorySaver)       Store (InMemoryStore)
─────────────────────────        ──────────────────────────────────
Per thread_id conversation        Cross-thread, per-user-id key-value
Conversation history              User preferences, saved facts
Lost between threads              Persists across threads
```

| Backend | Use case |
|---------|----------|
| `InMemoryStore` | Development / testing (this section) |
| Custom `BaseStore` | Production: SAP HANA, Redis, Postgres |

**SAP BTP note:** There is no SAP-published HANA-backed `LangGraph Store` yet. For production you would subclass `BaseStore` and implement `put / get / search` using `langchain-hana`. The interface is straightforward.

### 6.1 The Store API

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Namespace = (category, user_id) — gives per-user isolation automatically
alice_ns = ('preferences', 'alice')
bob_ns   = ('preferences', 'bob')

# put(namespace, key, value_dict)
store.put(alice_ns, 'preferred_region', {'value': 'eu10'})
store.put(alice_ns, 'language',         {'value': 'English'})
store.put(bob_ns,   'preferred_region', {'value': 'us10'})

# get(namespace, key) → Item | None
item = store.get(alice_ns, 'preferred_region')
print(f'Alice region: {item.value["value"]}')

# search(namespace) → list[Item]
alice_prefs = store.search(alice_ns)
for p in alice_prefs:
    print(f'  {p.key}: {p.value["value"]}')

# Isolation: Bob cannot see Alice's data
bob_prefs = store.search(bob_ns)
print(f'\nAlice: {len(alice_prefs)} pref(s)  |  Bob: {len(bob_prefs)} pref(s)')

# delete(namespace, key)
store.delete(alice_ns, 'language')
print(f'After delete: Alice has {len(store.search(alice_ns))} pref(s)')

### 6.2 Memory-aware Agent Node

Nodes receive `config` as a second parameter (type `RunnableConfig`). We read `user_id` from `config['configurable']` and use it to namespace the store — giving each user their own isolated memory.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

long_term_store = InMemoryStore()


class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


def memory_node(state: ChatState, config: RunnableConfig) -> dict:
    'Agent node that reads and writes long-term memory keyed by user_id.'
    user_id   = config.get('configurable', {}).get('user_id', 'anonymous')
    namespace = ('memory', user_id)
    last_msg  = state['messages'][-1].content

    if 'remember' in last_msg.lower():
        existing = long_term_store.search(namespace)
        key = f'fact_{len(existing) + 1}'
        long_term_store.put(namespace, key, {'text': last_msg})
        reply = f'Saved to long-term memory: "{last_msg}"'

    elif any(kw in last_msg.lower() for kw in ['recall', 'know about me', 'what do you know']):
        memories = long_term_store.search(namespace)
        if memories:
            facts = [m.value['text'] for m in memories]
            reply = 'Here is what I remember:\n' + '\n'.join(f'  - {f}' for f in facts)
        else:
            reply = 'I have no memories for you yet. Tell me something to remember!'

    else:
        count = len(long_term_store.search(namespace))
        reply = f'(Mocked LLM response. Long-term facts stored for you: {count})'

    return {'messages': [AIMessage(content=reply)]}


mem_builder = StateGraph(ChatState)
mem_builder.add_node('agent', memory_node)
mem_builder.add_edge(START, 'agent')
mem_builder.add_edge('agent', END)

memory_graph = mem_builder.compile(checkpointer=MemorySaver())
print('Memory graph compiled.')

In [ ]:
# Turn 1 — Alice saves a fact (thread-001)
alice_t1 = {'configurable': {'thread_id': 'thread-001', 'user_id': 'alice'}}
r = memory_graph.invoke(
    {'messages': [HumanMessage(content='Please remember: my SAP BTP region is eu10.')]},
    config=alice_t1,
)
print('Alice (thread-001):', r['messages'][-1].content)

# Turn 2 — NEW thread, same user → long_term_store persists the memory
alice_t2 = {'configurable': {'thread_id': 'thread-002', 'user_id': 'alice'}}
r = memory_graph.invoke(
    {'messages': [HumanMessage(content='What do you know about me?')]},
    config=alice_t2,   # different thread_id — MemorySaver has no history here
)
print('Alice (thread-002):', r['messages'][-1].content)  # recalls via long_term_store

# Bob — isolated namespace, cannot see Alice's facts
bob_t1 = {'configurable': {'thread_id': 'thread-003', 'user_id': 'bob'}}
r = memory_graph.invoke(
    {'messages': [HumanMessage(content='What do you know about me?')]},
    config=bob_t1,
)
print('Bob   (thread-003):', r['messages'][-1].content)  # no memories

print()
print('Key distinction:')
print('  MemorySaver (checkpointer) → per-thread conversation history')
print('  InMemoryStore (store)      → per-user long-term facts, cross-thread')

---
## Section 7 — Reflection / Self-critique Loop

A **reflection loop** adds an automated quality-control gate before an agent commits an action:

```
START → generate → critique ──[approved]──► END
                       │
                  [failed, retry]
                       │
                  back to generate (with the critique)
```

**SAP use case:** before writing data back to SAP (create purchase order, update ticket, send email), have a critic verify the draft:
- Is the currency specified?
- Is a requester email present?
- Does the amount look reasonable?

This is a **soft Human-in-the-Loop** — automated quality control without needing a human approver for every action. A `MAX_ITERATIONS` guard prevents infinite loops; on max-out you can route to HITL instead of forcing approval.

In [ ]:
class ReflectionState(TypedDict):
    messages:   Annotated[List[BaseMessage], add_messages]
    draft:      str
    critique:   str
    iterations: int
    approved:   bool


MAX_ITERATIONS = 3


def generate(state: ReflectionState) -> dict:
    iteration = state.get('iterations', 0) + 1
    critique  = state.get('critique', '')

    # Mock LLM: quality improves each iteration
    if iteration == 1:
        draft = 'Create order for ACME. Amount TBD.'                                           # bad
    elif iteration == 2:
        draft = 'Create order SO-NEW for ACME Corp. Amount: 5,000.'                            # missing currency
    else:
        draft = 'Create order SO-NEW for ACME Corp. Amount: 5,000 EUR. Requester: user@acme.com.'  # good

    tag = f'(revised — {critique[:40]})' if critique else '(first draft)'
    print(f'[generate] iteration {iteration} {tag}')
    return {'draft': draft, 'iterations': iteration, 'critique': ''}


def critique_draft(state: ReflectionState) -> dict:
    draft  = state['draft']
    issues = []

    if not any(cur in draft.upper() for cur in ['EUR', 'USD', 'GBP']):
        issues.append('missing currency')
    if '@' not in draft:
        issues.append('missing requester email')
    if not any(c.isdigit() for c in draft):
        issues.append('no amount specified')

    if issues:
        critique = 'Issues: ' + ', '.join(issues)
        print(f'[critique] FAIL — {critique}')
        return {'critique': critique, 'approved': False}

    print('[critique] APPROVED')
    return {'critique': '', 'approved': True}


def should_continue(state: ReflectionState) -> str:
    if state['approved']:
        return 'done'
    if state['iterations'] >= MAX_ITERATIONS:
        print('[router] max iterations reached — forcing approval (route to HITL in production)')
        return 'done'
    return 'retry'


builder = StateGraph(ReflectionState)
builder.add_node('generate', generate)
builder.add_node('critique', critique_draft)
builder.add_edge(START, 'generate')
builder.add_edge('generate', 'critique')
builder.add_conditional_edges(
    'critique',
    should_continue,
    {'retry': 'generate', 'done': END}
)

reflection_graph = builder.compile()
print(reflection_graph.get_graph().draw_ascii())

In [ ]:
result = reflection_graph.invoke({
    'messages':   [HumanMessage(content='Create a new sales order for ACME Corp.')],
    'draft':      '',
    'critique':   '',
    'iterations': 0,
    'approved':   False,
})

print('\n=== Final approved draft ===')
print(result['draft'])
print(f'\nApproved: {result["approved"]}  |  Iterations: {result["iterations"]}')

---
## Section 8 — Observability with Langfuse

In production you need to understand:
- Which nodes ran, and in what order?
- How long did each LLM call take?
- What was the exact prompt and response?
- Where did costs / tokens come from?

### Why Langfuse over LangSmith for SAP BTP?

| | LangSmith | Langfuse |
|---|-----------|----------|
| Hosting | SaaS only | **Self-hostable on BTP CF** or free cloud |
| Data sovereignty | Trace data leaves your tenant | Data stays in your BTP account |
| Cost | Paid tiers | Open-source, free self-hosted |
| SAP community | Mentioned as option | **Preferred for BTP** (data sovereignty) |

Langfuse is explicitly recommended in the SAP community for BTP because trace data (which may include business-sensitive prompts and responses) never leaves your tenant.

### Setup

**Option A — Free cloud** (easy, good for learning):  
Sign up at https://cloud.langfuse.com → create a project → copy public + secret key.

**Option B — Self-hosted on BTP** (production):  
Deploy Langfuse as a Cloud Foundry app on your BTP subaccount.

### Integration

Langfuse plugs in via a LangChain **callback**. You pass a `CallbackHandler` inside the `config` dict on every `invoke()` — **zero changes to graph code**.

In [ ]:
import os

try:
    from langfuse.callback import CallbackHandler as LangfuseHandler
    LANGFUSE_OK = True
    print('Langfuse installed.')
except ImportError:
    LANGFUSE_OK = False
    print('Langfuse not installed — run: pip install langfuse')
    print('Integration pattern is shown below regardless.')


# ── Credentials ────────────────────────────────────────────────────────────
# Set env vars before running (never hardcode keys):
#
#   LANGFUSE_PUBLIC_KEY = "pk-lf-..."
#   LANGFUSE_SECRET_KEY = "sk-lf-..."
#   LANGFUSE_HOST       = "https://cloud.langfuse.com"  (or BTP CF app URL)
#
# For SAP BTP self-hosted:
#   LANGFUSE_HOST = "https://langfuse.<your-cf-app>.cfapps.<region>.hana.ondemand.com"


def make_config(thread_id: str, user_id: str = 'sap-learner') -> dict:
    'Build a LangGraph config dict with Langfuse tracing attached.'
    base = {'configurable': {'thread_id': thread_id}}

    if not LANGFUSE_OK:
        return base

    handler = LangfuseHandler(
        public_key=os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-placeholder'),
        secret_key=os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-placeholder'),
        host=os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com'),
        session_id=thread_id,
        user_id=user_id,
        tags=['notebook-2', 'sap-btp'],
    )
    return {**base, 'callbacks': [handler]}


cfg = make_config('demo-trace-001')
print('Config:', {k: v for k, v in cfg.items() if k != 'callbacks'})
print(f'Callbacks: {len(cfg.get("callbacks", []))} handler(s) attached')

In [ ]:
# ── Zero-code-change integration ───────────────────────────────────────────
# Attach tracing to any graph by passing config — graph code is unchanged.
# Example: trace the reflection graph from Section 7.

cfg = make_config('reflection-trace-001')

result = reflection_graph.invoke(
    {
        'messages':   [HumanMessage(content='Create a sales order for ACME Corp for 8,000 EUR.')],
        'draft':      '',
        'critique':   '',
        'iterations': 0,
        'approved':   False,
    },
    config=cfg,   # ← Langfuse fires automatically on every LLM call
)

print('Final draft:', result['draft'])
print()

# ── What Langfuse captures per run ─────────────────────────────────────────
#
#  Trace: reflection_graph.invoke
#  ├── Span: generate (node, iteration 1)
#  │   └── Generation: llm.invoke  ← exact prompt · response · tokens · latency
#  ├── Span: critique (node)
#  ├── Span: generate (node, iteration 2)
#  │   └── Generation: llm.invoke
#  └── Span: critique (node, approved)
#
#  Metrics captured automatically:
#    - Total tokens (input + output) per trace
#    - Cost estimate (if model pricing is configured)
#    - End-to-end latency
#    - Per-node latency
#    - Error rate
#
#  Custom metadata:
#    LangfuseHandler(metadata={'sap_system': 'S4HANA', 'tenant': 'my-btp-subaccount'})
#
#  SAP BTP production checklist:
#    1. Deploy Langfuse as a CF app on your BTP subaccount
#    2. Set LANGFUSE_HOST to your CF app URL
#    3. Inject LANGFUSE_* keys as CF env vars (use BTP credential store, not hardcoded)
#    4. Pass make_config() into every agent.invoke() call

print('If Langfuse is configured → view trace at: https://cloud.langfuse.com')

---
## Summary & Next Steps

### What you learned in Notebook 2

| Concept | LangGraph API | SAP Use Case |
|---------|---------------|--------------|
| Supervisor pattern | `StateGraph` + conditional routing + worker wrappers | Orchestrate procurement · maintenance · HR agents |
| Subgraphs | `compile()` embedded as a node + wrapper function | Reusable SAP domain modules |
| Parallel execution | `Send` + `add_conditional_edges(START, fan_out, [...])` | Fan-out to multiple SAP systems simultaneously |
| Custom reducers | `Annotated[T, my_fn]` | Merge multi-agent results with business logic |
| Input/output schemas | `StateGraph(Full, input=In, output=Out)` | Clean public API · hide scratch fields |
| RAG tool node | `MockHanaDB` → `HanaDB` (langchain-hana) → `@tool` | HANA Cloud Vector Engine for grounding |
| Long-term memory | `InMemoryStore` · `put / get / search` · `user_id` namespace | Persist user preferences across sessions |
| Reflection loop | `generate → critique → conditional_edge` | Quality gate before writing to SAP |
| Observability | `LangfuseHandler` in `config['callbacks']` | Production tracing · cost tracking · audit |

### Notebook 3 roadmap (planned)

1. **MCP server for CAP** — call CAP OData services as structured agent tools (new SAP capability, 2025)
2. **Streaming responses** — token-by-token output for Fiori / SAP Build Apps UIs
3. **Cloud Foundry deployment** — package the agent as a CF app with health checks and CF env var injection
4. **End-to-end SAP workflow** — procurement agent: search HANA → create order → notify via email

### Key references

- [SAP AI Core Agent QuickLaunch Series](https://community.sap.com/t5/technology-blog-posts-by-sap/sap-ai-core-agent-quicklaunch-series-part0-prologue/ba-p/14104823)
- [Multi AI Agents — SAP Maintenance Notification](https://community.sap.com/t5/artificial-intelligence-and-machine-learning-blogs/multi-ai-agents-use-case-sap-maintenance-notification-creation/ba-p/13608309)
- [langchain-hana (official SAP)](https://github.com/SAP/langchain-integration-for-sap-hana-cloud)
- [Langfuse docs](https://langfuse.com/docs)
- [LangGraph multi-agent concepts](https://langchain-ai.github.io/langgraph/concepts/multi_agent/)
- [LangGraph Send API (map-reduce)](https://langchain-ai.github.io/langgraph/how-tos/map-reduce/)